<a href="https://colab.research.google.com/github/MelissaRegoRodrigues/FIELLI/blob/main/CNN_Otimizadores_Compara%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.image import load_img, img_to_array

#Análise da base escolhida

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

dataset_path = kagglehub.dataset_download("alessandrasala79/ai-vs-human-generated-dataset")

print("Dataset downloaded to:", dataset_path)

import os
print("Files in dataset folder:", os.listdir(dataset_path))

file_path = os.path.join(dataset_path, "dataset-ai-real-image")

 23%|██▎       | 2.27G/9.76G [01:04<03:32, 37.8MB/s]

depois de muita luta pra baixar a base de dados, fazendo o pré-processamento

In [ ]:
train_csv = pd.read_csv(os.path.join(dataset_path, "train.csv"))
test_csv  = pd.read_csv(os.path.join(dataset_path, "test.csv"))

train_images_dir = os.path.join(dataset_path, "train_data")
test_images_dir  = os.path.join(dataset_path, "test_data_v2")

In [ ]:
print("Arquivos na pasta de treino:")
print(os.listdir(test_images_dir)[:20])

In [ ]:
print("Arquivos na pasta de teste:")
print(os.listdir(test_images_dir)[:20])

In [ ]:
print(train_csv.head())

In [ ]:
train_csv= train_csv.drop(train_csv.columns[0], axis=1)
print(train_csv.head())

In [ ]:
train_csv['file_name'] = train_csv['file_name'].str.replace('train_data/', '', regex=False)
print(train_csv.head())

In [ ]:
print(test_csv.head())

In [ ]:
test_csv['id'] = test_csv['id'].str.replace('test_data_v2/', '', regex=False)
print(test_csv.head())

In [ ]:
train_csv['label'] = train_csv['label'].astype(str) # tinha q ta em string pro seguinte

In [ ]:
train_data, val_data = train_test_split(train_csv, test_size=0.25, random_state=42)

eu vi que no exemplo do tensorflow e do colleb do class já tão lidando com bases prontas pra isso (o cnn), nesse caso eu peguei do kaggle e elas tão em .png e .jpge, então pesquisando vi que o ImageDataGenerator é usado pra carregar essas imagens, redimensionar elas e normalizar. Além de fazer data augmentation que é quando a gente expande as imagens de treino pra ter mais variações (tipo embaçada e não embaçada)

In [ ]:
IMG_SIZE = (96,96)  # mudar se o colab n tankar esse tamanho tbm
BATCH_SIZE = 128

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=(0.8,1.0),
    horizontal_flip=True
)

# separando

val_datagen = ImageDataGenerator(rescale=1.0/255)
test_datagen = ImageDataGenerator(rescale=1.0/255)

# generator

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_data,
    directory=train_images_dir,
    x_col='file_name',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_data,
    directory=train_images_dir,
    x_col='file_name',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)


test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_csv,
    directory=test_images_dir,
    x_col='id',
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# Modelo CNN

In [ ]:
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(96, 96, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model


In [ ]:
optimizers = {
    'SGD': SGD(learning_rate = 0.01),
    'SGD with Momentum': SGD(momentum=0.9),
    'SGD Nesterov': SGD(momentum=0.9, nesterov=True),
    'Adagrad': Adagrad(),
    'RMSprop': RMSprop(),
    'Adam': Adam()
}

In [ ]:
print(train_data['label'].value_counts())


In [ ]:
print(test_csv)

In [ ]:
history_dict = {}
test_predictions_dict = {}
for name, optimizer in optimizers.items():
    model = create_model()
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'], batch_size=128)
    print(f"Treinando com {name}...")
    history = model.fit(train_generator, epochs=10, validation_data=val_generator, verbose=2)
    history_dict[name] = history.history
    #Avaliando no teste
    preds = model.predict(test_generator, verbose=1)
    pred_labels = (preds > 0.5).astype(int).flatten()  # converte probabilidade para 0 ou 1

    test_predictions_dict[name] = pred_labels

In [ ]:
for name, history in history_dict.items():
    train_acc = history['accuracy'][-1]          # último epoch
    train_loss = history['loss'][-1]
    val_acc = history['val_accuracy'][-1]
    val_loss = history['val_loss'][-1]
    print(f"{name}:")
    print(f"  Train Accuracy: {train_acc:.4f}, Train Loss: {train_loss:.4f}")
    print(f"  Val Accuracy:   {val_acc:.4f}, Val Loss:   {val_loss:.4f}\n")


In [ ]:
plt.figure(figsize=(12, 8))
for name, history in history_dict.items():
    plt.plot(history['val_loss'], label=name)


plt.xlabel('Epochs')
plt.ylabel('Loss Acc')
plt.legend()
plt.grid()

In [ ]:
plt.figure(figsize=(12, 8))
for name, history in history_dict.items():
    plt.plot(history['val_accuracy'], label=name)


plt.xlabel('Epochs')
plt.ylabel('Validation Acc')
plt.legend()
plt.grid()

In [ ]:
for name, test_accuracy in test_accuracy_dict.items():
    print(f"{name}: {test_accuracy:.4f}")